# 6. 模型格式与序列化

高效的模型存储格式直接影响加载速度、内存占用和跨平台兼容性。在产业级部署中，模型从训练到推理往往需要经历多次格式转换，理解每种格式的设计哲学和适用场景是端侧部署工程师的核心能力。

## 什么是模型格式？

模型格式是模型权重、结构和元数据的序列化方式。核心格式包括：
- **PyTorch (.pt/.pth)**：Pickle序列化，灵活但不安全，加载慢
- **ONNX (.onnx)**：跨框架标准格式，Protobuf序列化，支持算子标准化
- **SafeTensors**：HuggingFace推出的安全格式，零拷贝mmap加载，无Pickle安全风险
- **GGUF**：llama.cpp使用的格式，支持多种量化类型，mmap零拷贝加载
- **TensorRT (.engine/.plan)**：NVIDIA推理引擎的优化格式，层融合+内核自动调优
- **TFLite (.tflite)**：Google端侧推理格式，FlatBuffers序列化，支持Android/嵌入式
- **Core ML (.mlpackage/.mlmodelc)**：Apple端侧格式，ANE优化，支持iOS/macOS

## 为什么模型格式重要？

1. **加载速度**：mmap零拷贝加载（SafeTensors/GGUF）比Pickle反序列化快10-100倍
2. **安全性**：Pickle反序列化可执行任意代码，SafeTensors避免了此风险
3. **内存效率**：量化格式（GGUF）直接加载低精度权重，无需先加载FP16再量化
4. **跨平台**：ONNX等标准格式使得模型可在不同框架和硬件间迁移
5. **推理性能**：TensorRT等引擎格式通过层融合和内核选择实现最优推理性能
6. **分发效率**：大模型（70B+）需要分片存储，格式设计直接影响下载和加载体验

## 核心原理

**mmap零拷贝**：操作系统将文件直接映射到进程地址空间，无需将数据从内核空间拷贝到用户空间。首次访问时触发缺页中断（page fault），操作系统按需加载对应页面：
$$t_{\text{mmap}} = O(1) \quad \text{（映射本身）}, \quad t_{\text{access}} = O(\text{实际访问的页面数})$$
vs 传统读取：$t_{\text{read}} = O(\text{file size})$（必须全部读入）

mmap的关键优势：
- **惰性加载**：只加载实际使用的权重，多LoRA场景下可按需加载不同适配器
- **跨进程共享**：多个进程可共享同一mmap映射，减少总内存占用
- **大模型支持**：70B模型无需全部加载到内存，按层按需访问

**Block量化（GGUF）**：将权重按block分组量化，每个block独立的scale和offset：
$$W_{q,k} = \text{round}(W_k / s_k), \quad s_k = \frac{\max(|W_k|)}{q_{\max}}$$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import struct
import os
import time
import hashlib
import mmap
import tempfile
import shutil
from pathlib import Path

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

try:
    import safetensors.torch as st
    print(f"SafeTensors: available")
except ImportError:
    print(f"SafeTensors: not installed (pip install safetensors)")

try:
    import onnxruntime as ort
    print(f"ONNX Runtime: {ort.__version__}")
except ImportError:
    print(f"ONNX Runtime: not installed (pip install onnxruntime)")

try:
    from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
    from cryptography.hazmat.primitives import padding as sym_padding
    from cryptography.hazmat.backends import default_backend
    print(f"Cryptography: available")
except ImportError:
    print(f"Cryptography: not installed (pip install cryptography)")

### GGUF格式详解

#### 什么是GGUF？

GGUF（GPT-Generated Unified Format）是llama.cpp的标准格式，支持元数据嵌入、多种量化类型（Q2_K到Q8_0）和mmap零拷贝加载。GGUF是当前端侧CPU推理（尤其是Apple Silicon和x86 AVX）最主流的格式。

#### 为什么GGUF有效？

1. **mmap零拷贝加载**：操作系统将文件直接映射到进程地址空间，加载时间$O(1)$
2. **K-Quant混合量化**：对权重矩阵的不同部分使用不同精度，精度-压缩比更优
3. **自包含格式**：词表、超参数、量化参数全部嵌入文件，单文件即可推理
4. **对齐要求**：数据按32字节对齐，适配AVX/SSE向量指令

#### GGUF文件结构

```
| Magic (4B) | Version (4B) | Tensor Count (8B) | Metadata KV Count (8B) |
| Metadata KV Pairs ... |
| Tensor Info (name + dims + type + offset) ... |
| Padding to Alignment |
| Tensor Data ... |
```

#### GGUF的Block量化原理

将权重按block分组量化：
$$W_{q,k} = \text{round}(W_k / s_k), \quad s_k = \frac{\max(|W_k|)}{q_{\max}}$$

K-Quant分层结构：
- super-block（256权重）：存储全局scale $s_{\text{super}}$
- sub-block（32权重）：存储局部scale $s_{\text{sub}}$和offset
- 最终量化值：$W_{q,i} = \text{round}(W_i / (s_{\text{super}} \cdot s_{\text{sub}}))$

K-Quant类型对比（7B模型典型指标）：
| 类型 | bits/weight | 文件大小 | Wiki2 PPL | 适用场景 |
|------|------------|---------|-----------|----------|
| Q8_0 | 8.5 | ~7.4GB | ~5.40 | 高精度端侧 |
| Q6_K | 6.59 | ~5.7GB | ~5.45 | 精度-大小均衡 |
| Q5_K_M | 5.69 | ~4.9GB | ~5.52 | 推荐默认选择 |
| Q4_K_M | 4.80 | ~4.1GB | ~5.60 | 端侧8GB内存 |
| Q3_K_M | 3.91 | ~3.4GB | ~5.82 | 端侧4GB内存 |
| Q2_K | 2.56 | ~2.2GB | ~6.45 | 极限压缩 |

In [ ]:
class GGUFFileSimulator:
    """GGUF文件格式模拟"""
    GGUF_MAGIC = 0x47475546  # 'GGUF' in little-endian
    TYPES = {0: 'F32', 1: 'F16', 2: 'Q4_0', 3: 'Q4_1', 6: 'Q5_0', 7: 'Q5_1',
             8: 'Q8_0', 12: 'Q4_K', 13: 'Q5_K', 14: 'Q6_K'}

    BYTES_PER_ELEMENT = {
        'F32': 4.0, 'F16': 2.0,
        'Q4_0': 18/32, 'Q4_1': 20/32,
        'Q5_0': 22/32, 'Q5_1': 24/32,
        'Q8_0': 34/32,
        'Q4_K': 144/256, 'Q5_K': 176/256, 'Q6_K': 210/256,
    }

    def __init__(self):
        self.metadata = {}
        self.tensor_infos = {}
        self.alignment = 32

    def add_metadata(self, key, value):
        self.metadata[key] = value

    def add_tensor_info(self, name, shape, dtype='Q4_K', offset=0):
        n_elements = 1
        for s in shape:
            n_elements *= s
        bpe = self.BYTES_PER_ELEMENT.get(dtype, 2)
        size_bytes = int(n_elements * bpe)
        self.tensor_infos[name] = {
            'shape': shape, 'dtype': dtype, 'offset': offset, 'size': size_bytes,
            'bits_per_weight': bpe * 8
        }

    def estimate_file_size(self):
        header_size = 64
        metadata_size = sum(len(k) + 16 for k in self.metadata)
        tensor_info_size = sum(len(k) + 64 for k in self.tensor_infos)
        tensor_data_size = sum(t['size'] for t in self.tensor_infos.values())
        return header_size + metadata_size + tensor_info_size + tensor_data_size

gguf = GGUFFileSimulator()
gguf.add_metadata('general.architecture', 'llama')
gguf.add_metadata('llm.context_length', 4096)
gguf.add_metadata('llm.embedding_length', 4096)
gguf.add_metadata('llm.layer_count', 32)
gguf.add_metadata('llm.attention.head_count', 32)

gguf.add_tensor_info('token_embd.weight', [4096, 32000], 'Q4_K')
for i in range(32):
    gguf.add_tensor_info(f'blk.{i}.attn_q.weight', [4096, 4096], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_k.weight', [4096, 512], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_v.weight', [4096, 512], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_output.weight', [4096, 4096], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_gate.weight', [4096, 11008], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_up.weight', [4096, 11008], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_down.weight', [11008, 4096], 'Q4_K')

print(f"=== GGUF格式模拟 (7B模型) ===")
print(f"元数据条目: {len(gguf.metadata)}")
print(f"张量条目: {len(gguf.tensor_infos)}")
print(f"\n元数据内容:")
for k, v in gguf.metadata.items():
    print(f"  {k}: {v}")

print(f"\n量化类型 bits/weight 对比:")
for dtype_name, bpe in sorted(GGUFFileSimulator.BYTES_PER_ELEMENT.items(), key=lambda x: x[1]):
    print(f"  {dtype_name:<8} {bpe*8:.2f} bits/weight")

est_size = gguf.estimate_file_size()
print(f"\nQ4_K估算文件大小: {est_size/1024/1024/1024:.2f} GB")
print(f"FP16原始大小: {7.0:.2f} GB")
print(f"压缩比: {7.0/est_size*1024*1024*1024:.1f}x")

print(f"\nGGUF核心特性:")
print(f"1. 单文件格式: 元数据+权重一体")
print(f"2. mmap支持: 零拷贝加载，多进程共享")
print(f"3. 丰富量化: Q4_0到Q6_K多种选择")
print(f"4. 扩展性: 键值对元数据，易于扩展")
print(f"5. 对齐: 32字节对齐，适配SIMD指令")

print(f"\n=== 产业级GGUF转换命令 ===")
print(f"# 1. HuggingFace模型 → GGUF (llama.cpp)")
print(f"python convert_hf_to_gguf.py /path/to/model --outtype f16 --outfile model-f16.gguf")
print(f"python convert_hf_to_gguf.py /path/to/model --outtype q4_k_m --outfile model-q4km.gguf")
print(f"")
print(f"# 2. GGUF量化 (llama-quantize)")
print(f"./llama-quantize model-f16.gguf model-q4km.gguf Q4_K_M")
print(f"./llama-quantize model-f16.gguf model-q5km.gguf Q5_K_M")
print(f"./llama-quantize model-f16.gguf model-q8_0.gguf Q8_0")
print(f"")
print(f"# 3. GGUF分片 (大模型>50GB)")
print(f"./llama-gguf-split --split --split-max-size 5G model-q4km.gguf model-q4km")
print(f"")
print(f"# 4. 读取GGUF元数据 (gguf库)")
print(f"from gguf import GGUFReader")
print(f"reader = GGUFReader('model-q4km.gguf')")
print(f"for key, val in reader.fields.items():")
print(f"    print(f'{key}: {val}')")
print(f"")
print(f"# 5. 推理 (llama-cli)")
print(f"./llama-cli -m model-q4km.gguf -p 'Hello' -n 128 --temp 0.7")

### SafeTensors格式

#### 什么是SafeTensors？

HuggingFace推出的安全模型存储格式，使用零拷贝mmap加载，无Pickle反序列化安全风险。已成为HuggingFace Hub的默认模型格式。

#### 为什么用SafeTensors？

1. **安全性**：Pickle反序列化可执行任意代码（`__reduce__`方法），SafeTensors使用纯数据格式，无代码执行风险
2. **加载速度**：mmap零拷贝加载比Pickle反序列化快10-100倍
3. **跨语言**：简单的二进制格式，可被任何编程语言读取（Rust/Go/JS/C++）
4. **惰性加载**：可只加载需要的张量，支持大模型按需加载
5. **分片支持**：大模型可拆分为多个分片文件（model-00001-of-00008.safetensors）

#### SafeTensors的文件结构

```
| Header Size (8 bytes, LE uint64) | JSON Header | Padding | Tensor Data |
```

- Header Size：头部大小，8字节无符号小端整数
- JSON Header：张量元数据（名称、数据类型、形状、偏移量），偏移量相对于数据区起始
- Padding：数据区按8字节对齐
- Tensor Data：连续存储的张量数据，每个张量按其dtype的自然对齐排列

加载时间对比：
- Pickle：$t = O(\text{file size})$（需反序列化+拷贝）
- SafeTensors：$t = O(1)$（mmap映射，按需加载）

#### PyTorch Pickle安全风险

```python
# 恶意.pt文件可以执行任意代码
import pickle
class Exploit:
    def __reduce__(self):
        return (os.system, ('rm -rf /',))
# torch.load('malicious.pt') 会执行上述命令
```
这就是为什么HuggingFace强制使用SafeTensors，且`torch.load`从PyTorch 2.0起默认`weights_only=True`。

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, dim=256, hidden_dim=1024):
        super().__init__()
        self.embed = nn.Embedding(32000, dim)
        self.layer0 = nn.Linear(dim, hidden_dim)
        self.layer1 = nn.Linear(hidden_dim, dim)
        self.output = nn.Linear(dim, 32000)
    def forward(self, x):
        x = self.embed(x)
        x = torch.relu(self.layer0(x))
        x = self.layer1(x)
        return self.output(x)

model = SimpleModel(dim=256, hidden_dim=1024)
state_dict = model.state_dict()

tmpdir = tempfile.mkdtemp()
pt_path = os.path.join(tmpdir, 'model.pt')
st_path = os.path.join(tmpdir, 'model.safetensors')

torch.save(state_dict, pt_path)

try:
    import safetensors.torch as st
    st.save_file(state_dict, st_path)
    has_st = True
except ImportError:
    has_st = False
    print("safetensors未安装，跳过实际save/load演示")

print(f"=== SafeTensors vs PyTorch Pickle 实际对比 ===")
pt_size = os.path.getsize(pt_path)
print(f"\n文件大小:")
print(f"  PyTorch (.pt):     {pt_size/1024:.1f} KB")
if has_st:
    st_size = os.path.getsize(st_path)
    print(f"  SafeTensors (.st): {st_size/1024:.1f} KB")
    print(f"  大小差异:          {(pt_size-st_size)/1024:.1f} KB (Pickle开销)")

print(f"\n--- 加载速度基准测试 ---")
n_runs = 50

t0 = time.perf_counter()
for _ in range(n_runs):
    loaded_pt = torch.load(pt_path, map_location='cpu', weights_only=True)
pt_load_ms = (time.perf_counter() - t0) / n_runs * 1000

if has_st:
    t0 = time.perf_counter()
    for _ in range(n_runs):
        loaded_st = st.load_file(st_path)
    st_load_ms = (time.perf_counter() - t0) / n_runs * 1000

    print(f"  PyTorch pickle加载: {pt_load_ms:.2f} ms")
    print(f"  SafeTensors加载:    {st_load_ms:.2f} ms")
    print(f"  加速比:              {pt_load_ms/st_load_ms:.1f}x")

    print(f"\n--- mmap零拷贝加载 ---")
    t0 = time.perf_counter()
    for _ in range(n_runs):
        with open(st_path, 'rb') as f:
            header_size = struct.unpack('<Q', f.read(8))[0]
            header_json = f.read(header_size).decode('utf-8')
            header = json.loads(header_json)
    mmap_load_ms = (time.perf_counter() - t0) / n_runs * 1000
    print(f"  仅解析header: {mmap_load_ms:.2f} ms (O(1)映射，数据按需加载)")
    print(f"  Header内容示例:")
    for i, (name, meta) in enumerate(header.items()):
        if i >= 2:
            break
        print(f"    {name}: dtype={meta['dtype']}, shape={meta['shape']}, offsets={meta['data_offsets']}")

    print(f"\n--- 惰性加载演示 ---")
    loaded_st = st.load_file(st_path)
    for name, tensor in loaded_st.items():
        match = torch.allclose(tensor, state_dict[name])
        print(f"  {name}: shape={list(tensor.shape)}, 一致性={'✓' if match else '✗'}")

    print(f"\n--- 大模型分片策略 ---")
    shard_size = 5 * 1024 * 1024 * 1024
    total_params = sum(p.numel() for p in model.parameters())
    total_bytes = total_params * 2
    n_shards = max(1, (total_bytes + shard_size - 1) // shard_size)
    print(f"  模型参数量: {total_params/1e6:.1f}M")
    print(f"  FP16大小: {total_bytes/1024/1024:.1f} MB")
    print(f"  5GB分片: {n_shards} 个文件")
    print(f"  文件命名: model-00001-of-{n_shards:05d}.safetensors")
    print(f"  索引文件: model.safetensors.index.json (记录每个张量所在分片)")
else:
    print(f"  请安装safetensors: pip install safetensors")

shutil.rmtree(tmpdir, ignore_errors=True)

print(f"\n=== SafeTensors核心优势 ===")
print(f"1. 安全性: 无pickle反序列化风险（无__reduce__攻击面）")
print(f"2. 加载速度: mmap零拷贝，比pickle快10-100x")
print(f"3. 惰性加载: 可只加载需要的张量（data_offsets定位）")
print(f"4. 跨框架: HuggingFace标准格式，Rust/Go/JS均可读取")
print(f"5. 分片: 大模型可拆分为多个safetensors文件")
print(f"6. 产业实践: HuggingFace Hub已默认使用safetensors格式")

### ONNX格式与跨框架导出

#### 什么是ONNX？

ONNX（Open Neural Network Exchange）是跨框架的模型表示标准，使用Protobuf序列化。ONNX定义了标准算子集（opset）和计算图表示，使得模型可以在PyTorch、TensorFlow、MXNet等框架间迁移。

#### 为什么ONNX重要？

1. **推理引擎入口**：TensorRT、OpenVINO、ONNX Runtime等推理引擎都以ONNX作为输入格式
2. **硬件适配**：不同硬件厂商（NVIDIA/Intel/Qualcomm/ARM）都支持ONNX算子
3. **算子标准化**：opset版本确保算子语义一致，避免框架间差异
4. **图优化基础**：ONNX图可进行常量折叠、死代码消除、算子融合等优化

#### ONNX导出的关键概念

- **动态轴（dynamic_axes）**：指定哪些维度是动态的（如batch_size、seq_len）
- **opset版本**：不同opset版本支持不同算子，推荐使用opset 14+
- **ONNX检查**：`onnx.checker.check_model()`验证模型合法性
- **自定义算子**：如果模型使用了ONNX不支持的算子，需要注册自定义算子或使用`torch.onnx.export`的`custom_opsets`

In [ ]:
class SimpleTransformerBlock(nn.Module):
    """用于ONNX导出演示的简化Transformer块"""
    def __init__(self, dim=64, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return x

model = SimpleTransformerBlock(dim=64, n_heads=4)
model.eval()

tmpdir = tempfile.mkdtemp()
onnx_path = os.path.join(tmpdir, 'simple_transformer.onnx')

dummy_input = torch.randn(1, 16, 64)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    opset_version=17,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size', 1: 'seq_len'},
        'output': {0: 'batch_size', 1: 'seq_len'},
    },
)

import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print(f"=== ONNX导出 ===")
print(f"模型: SimpleTransformerBlock")
print(f"opset版本: 17")
print(f"文件大小: {os.path.getsize(onnx_path)/1024:.1f} KB")
print(f"输入: input [batch_size, seq_len, 64]")
print(f"输出: output [batch_size, seq_len, 64]")
print(f"动态轴: batch_size, seq_len")

print(f"\n--- ONNX计算图信息 ---")
print(f"节点数: {len(onnx_model.graph.node)}")
print(f"算子类型: {set(n.op_type for n in onnx_model.graph.node)}")

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

    test_inputs = [
        torch.randn(1, 16, 64),
        torch.randn(2, 32, 64),
        torch.randn(1, 1, 64),
    ]

    print(f"\n--- ONNX Runtime推理验证 ---")
    for i, test_input in enumerate(test_inputs):
        with torch.no_grad():
            pt_output = model(test_input).numpy()
        ort_output = sess.run(None, {'input': test_input.numpy()})[0]
        cos_sim = np.sum(pt_output * ort_output) / (np.linalg.norm(pt_output) * np.linalg.norm(ort_output))
        max_diff = np.max(np.abs(pt_output - ort_output))
        print(f"  shape={list(test_input.shape)}: cos_sim={cos_sim:.6f}, max_diff={max_diff:.2e}")

    print(f"\n--- 动态shape验证 ---")
    print(f"  ✓ batch_size和seq_len均可动态变化")
    print(f"  ✓ PyTorch与ONNX Runtime输出余弦相似度>0.9999")
except ImportError:
    print(f"\nONNX Runtime未安装，跳过推理验证 (pip install onnxruntime)")

print(f"\n--- 产业级ONNX导出注意事项 ---")
print(f"1. 动态轴: 必须指定batch和seq_len为动态，否则推理时只能用固定shape")
print(f"2. opset: 推荐opset 14+，支持更多算子（如GELU、LayerNorm）")
print(f"3. 验证: onnx.checker.check_model() + onnxruntime推理结果对比")
print(f"4. 优化: onnxoptimizer常量折叠+死代码消除后再送入推理引擎")
print(f"5. 量化: onnxruntime.quantization支持训练后量化(PTQ)")
print(f"6. 精度验证: 余弦相似度>0.999，max_diff<1e-5为合格")

shutil.rmtree(tmpdir, ignore_errors=True)

### 产业级推理引擎格式

#### TensorRT (.engine/.plan)

NVIDIA GPU推理的终极格式。TensorRT将ONNX模型编译为高度优化的引擎：
- **层融合**：Conv+BN+ReLU融合为单个kernel，减少显存访问
- **内核自动调优**：在目标GPU上实测选择最优kernel实现
- **精度校准**：INT8量化时使用校准数据集确定最优量化参数
- **动态batch**：支持优化配置文件（optimal profile）指定min/opt/max shape

编译流程：`ONNX → TensorRT Builder → .engine`

注意：`.engine`文件与GPU架构绑定，A100编译的引擎无法在4090上使用。

#### TFLite (.tflite)

Google端侧推理格式，使用FlatBuffers序列化（零拷贝、无需解析）：
- 支持INT8/INT4量化、FP16推理
- 支持Android NNAPI委托、GPU委托、Edge TPU委托
- 转换流程：`Keras/TF → TFLite Converter → .tflite`

#### Core ML (.mlpackage/.mlmodelc)

Apple端侧格式，针对Apple Silicon优化：
- 自动调度到ANE（Apple Neural Engine）、GPU、CPU
- 支持INT4/INT8量化、palette量化
- 转换流程：`PyTorch → coremltools → .mlpackage`

#### QNN Context Binary

高通AI Engine Direct (QNN)的编译格式：
- 针对Hexagon DSP/NPU优化
- 支持INT8/INT4量化推理
- 转换流程：`ONNX → qnn-converter → qnn-context-binary`

In [ ]:
print(f"=== 产业级模型格式对比 ===")
formats = [
    ('GGUF', 'llama.cpp', 'Block量化+mmap', '端侧CPU推理(Apple Silicon/x86)', 'Q2_K~Q8_0'),
    ('SafeTensors', 'HuggingFace', '安全+mmap+分片', '模型分发/加载/多LoRA', 'FP16/BF16'),
    ('ONNX', '跨框架标准', '标准算子+图优化', '推理引擎输入格式', 'FP32/FP16/INT8'),
    ('TensorRT', 'NVIDIA', '层融合+内核调优', 'NVIDIA GPU推理', 'FP16/INT8/INT4'),
    ('TFLite', 'Google', 'FlatBuffers+委托', 'Android/嵌入式端侧', 'FP32/FP16/INT8'),
    ('Core ML', 'Apple', 'ANE优化+palette', 'iOS/macOS端侧', 'FP16/INT4/INT8'),
    ('QNN Binary', 'Qualcomm', 'Hexagon NPU指令', '骁龙端侧推理', 'INT8/INT4'),
]
print(f"\n{'格式':<15} {'生态':<18} {'特点':<20} {'适用场景':<28} {'精度支持':<18}")
print("-" * 99)
for name, eco, feature, scene, precision in formats:
    print(f"{name:<15} {eco:<18} {feature:<20} {scene:<28} {precision:<18}")

print(f"\n=== 产业级格式转换工作流 ===")
workflows = [
    ('训练 → CPU端侧', 'PyTorch → ONNX → GGUF (llama.cpp convert)', '量化+mmap加载'),
    ('训练 → NVIDIA GPU', 'PyTorch → ONNX → TensorRT (trtexec)', '层融合+内核调优'),
    ('训练 → Android', 'PyTorch → ONNX → TFLite (ai-edge-litert)', 'NNAPI/GPU委托'),
    ('训练 → iOS', 'PyTorch → Core ML (coremltools)', 'ANE自动调度'),
    ('训练 → 骁龙NPU', 'PyTorch → ONNX → QNN (qnn-converter)', 'Hexagon DSP优化'),
    ('HuggingFace分发', 'PyTorch → SafeTensors (safetensors库)', '安全+mmap加载'),
    ('多LoRA服务', 'SafeTensors基座 + 多个LoRA适配器', '按需切换适配器'),
]
print(f"\n{'场景':<22} {'转换路径':<50} {'关键优势':<20}")
print("-" * 92)
for scene, path, advantage in workflows:
    print(f"{scene:<22} {path:<50} {advantage:<20}")

print(f"\n=== 格式选择决策树 ===")
print(f"1. 目标硬件是什么?")
print(f"   ├─ NVIDIA GPU → TensorRT (最优性能)")
print(f"   ├─ Apple Silicon → GGUF (CPU) / Core ML (ANE)")
print(f"   ├─ Android → TFLite (NNAPI/GPU委托)")
print(f"   ├─ 骁龙NPU → QNN Context Binary")
print(f"   └─ 通用CPU → GGUF (llama.cpp)")
print(f"2. 是否需要跨框架?")
print(f"   └─ 是 → ONNX (中间表示)")
print(f"3. 是否需要安全分发?")
print(f"   └─ 是 → SafeTensors (无Pickle风险)")
print(f"4. 是否需要多LoRA?")
print(f"   └─ 是 → SafeTensors基座 + LoRA适配器 (按需mmap加载)")

---
### 模型分片与大模型加载

70B+参数的模型无法存储在单个文件中（文件系统限制、内存限制、下载效率），需要分片存储和按需加载。

#### 为什么需要分片？

1. **文件系统限制**：FAT32最大4GB，ext4最大16TB，但单个大文件下载和校验困难
2. **内存限制**：70B FP16模型约140GB，无法一次性加载到大多数设备
3. **下载效率**：分片支持并行下载和断点续传
4. **按需加载**：推理时只加载当前需要的层，减少内存占用

#### 分片策略

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **均匀分片** | 按参数量均匀切割为N片 | 下载均衡 | 层可能被切断 |
| **按层分片** | 每N层一个分片 | 层完整性 | 分片大小不均 |
| **按组件分片** | Attention/FFN/Embedding分开 | 按需加载 | 管理复杂 |

#### HuggingFace分片规范

```
model-00001-of-00008.safetensors  (5GB)
model-00002-of-00008.safetensors  (5GB)
...
model-00008-of-00008.safetensors  (3.2GB)
model.safetensors.index.json      (元数据索引)
```

索引文件记录每个张量所在的分片：
```json
{"metadata": {"total_size": 13476836352},
 "weight_map": {
   "model.embed_tokens.weight": "model-00001-of-00008.safetensors",
   "model.layers.0.self_attn.q_proj.weight": "model-00001-of-00008.safetensors",
   ...
 }}
```

#### 分片加载延迟模型

$$T_{\text{load}} = T_{\text{index}} + \max_{i} T_{\text{shard},i}^{\text{parallel}} + T_{\text{verify}}$$

其中$T_{\text{index}}$为索引加载时间，$T_{\text{shard},i}$为第$i$个分片的加载时间，$T_{\text{verify}}$为哈希校验时间。

In [ ]:
class ModelSharding:
    """模型分片与完整性校验"""
    def __init__(self, shard_size_bytes=5*1024*1024*1024):
        self.shard_size = shard_size_bytes

    @staticmethod
    def compute_sha256(filepath: str) -> str:
        h = hashlib.sha256()
        with open(filepath, 'rb') as f:
            while True:
                chunk = f.read(8*1024*1024)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()

    def shard_model(self, src_path: str, output_dir: str) -> dict:
        os.makedirs(output_dir, exist_ok=True)
        src_size = os.path.getsize(src_path)
        n_shards = max(1, (src_size + self.shard_size - 1) // self.shard_size)
        weight_map = {}
        shard_hashes = {}

        with open(src_path, 'rb') as fin:
            for i in range(n_shards):
                shard_name = f"model-{i+1:05d}-of-{n_shards:05d}.bin"
                shard_path = os.path.join(output_dir, shard_name)
                remaining = min(self.shard_size, src_size - i * self.shard_size)
                with open(shard_path, 'wb') as fout:
                    written = 0
                    while written < remaining:
                        chunk = fin.read(min(8*1024*1024, remaining - written))
                        if not chunk:
                            break
                        fout.write(chunk)
                        written += len(chunk)
                shard_hash = self.compute_sha256(shard_path)
                shard_hashes[shard_name] = shard_hash
                weight_map[f"shard_{i}"] = shard_name

        index = {
            'metadata': {'total_size': src_size, 'n_shards': n_shards},
            'weight_map': weight_map,
            'shard_hashes': shard_hashes,
        }
        index_path = os.path.join(output_dir, 'model.bin.index.json')
        with open(index_path, 'w') as f:
            json.dump(index, f, indent=2)
        return index

    @staticmethod
    def verify_shards(index_path: str) -> dict:
        with open(index_path) as f:
            index = json.load(f)
        results = {'valid': True, 'shards': {}}
        shard_dir = os.path.dirname(index_path)
        for shard_name, expected_hash in index['shard_hashes'].items():
            shard_path = os.path.join(shard_dir, shard_name)
            actual_hash = ModelSharding.compute_sha256(shard_path)
            match = actual_hash == expected_hash
            results['shards'][shard_name] = {'match': match, 'size': os.path.getsize(shard_path)}
            if not match:
                results['valid'] = False
        return results

print("=== 模型分片实战 ===")

tmpdir = tempfile.mkdtemp()
model_path = os.path.join(tmpdir, 'large_model.bin')
model_data = os.urandom(10 * 1024 * 1024)
with open(model_path, 'wb') as f:
    f.write(model_data)

sharder = ModelSharding(shard_size_bytes=3*1024*1024)
shard_dir = os.path.join(tmpdir, 'shards')
index = sharder.shard_model(model_path, shard_dir)

print(f"原始大小: {index['metadata']['total_size']/1024/1024:.1f} MB")
print(f"分片数: {index['metadata']['n_shards']}")
print(f"\n分片详情:")
for shard_name, shard_hash in index['shard_hashes'].items():
    shard_size = os.path.getsize(os.path.join(shard_dir, shard_name))
    print(f"  {shard_name}: {shard_size/1024/1024:.1f} MB, SHA256={shard_hash[:16]}...")

verify = ModelSharding.verify_shards(os.path.join(shard_dir, 'model.bin.index.json'))
print(f"\n完整性校验: {'✓ 全部通过' if verify['valid'] else '✗ 校验失败'}")

corrupted_path = os.path.join(shard_dir, list(index['shard_hashes'].keys())[0])
with open(corrupted_path, 'r+b') as f:
    f.write(b'\x00\x00\x00\x00')
verify_corrupt = ModelSharding.verify_shards(os.path.join(shard_dir, 'model.bin.index.json'))
print(f"篡改检测: {'✓ 检测到篡改' if not verify_corrupt['valid'] else '✗ 未检测到'}")
for name, info in verify_corrupt['shards'].items():
    if not info['match']:
        print(f"  篡改文件: {name}")

shutil.rmtree(tmpdir, ignore_errors=True)

print(f"\n=== 大模型分片策略分析 ===")
for model_name, params_b, dtype in [('7B FP16', 7, 2), ('70B FP16', 70, 2), ('70B INT4', 70, 0.5)]:
    total_size = params_b * 1e9 * dtype
    shard_size = 5 * 1024 * 1024 * 1024
    n_shards = max(1, (total_size + shard_size - 1) // shard_size)
    print(f"  {model_name}: {total_size/1e9:.1f} GB → {n_shards}个分片(5GB/片)")

print(f"\n=== 产业实践要点 ===")
print(f"1. HuggingFace默认5GB分片，兼顾下载效率和文件管理")
print(f"2. SHA256校验确保分片完整性，防止损坏/篡改模型")
print(f"3. mmap加载时，OS按需分页，实际只需加载使用的层")
print(f"4. 并行下载可显著加速大模型加载，但受限于磁盘带宽")
print(f"5. GGUF也支持分片: ./llama-gguf-split --split --split-max-size 5G")
print(f"6. 索引文件(model.safetensors.index.json)记录每个张量所在分片")

---
### 模型加密与知识产权保护

在端侧部署中，模型文件存储在用户设备上，存在被提取、逆向和复制的风险。模型加密是保护知识产权的关键手段。

#### 威胁模型

| 攻击方式 | 难度 | 防御手段 |
|---------|------|----------|
| **直接复制模型文件** | 低 | 文件加密 + DRM |
| **内存dump提取权重** | 中 | 运行时解密 + 内存保护 |
| **逆向推理算法** | 高 | 模型混淆 + 算法白盒化 |
| **侧信道攻击** | 很高 | 常数时间实现 + 功耗随机化 |

#### 加密方案对比

| 方案 | 原理 | 安全性 | 性能开销 | 适用场景 |
|------|------|--------|---------|----------|
| **AES-256加密权重** | 权重文件加密，推理前解密 | 中 | 加载时+10-30% | 通用保护 |
| **运行时解密** | 仅解密当前使用的层 | 中高 | 推理时+5-15% | 内存受限设备 |
| **白盒加密** | 密钥嵌入推理代码，混淆密钥 | 低中 | 推理时+20-50% | 防逆向工程 |
| **平台DRM** | 利用硬件安全模块 | 高 | 极低 | iOS/Android |
| **模型混淆** | 权重重排+冗余计算 | 低 | 推理时+10-30% | 辅助手段 |

#### 平台DRM方案

- **iOS**: Keychain + Data Protection API + Secure Enclave
  - 模型密钥存储在Secure Enclave中，App无法直接读取
  - Core ML模型可绑定设备，防止跨设备复制
- **Android**: Android Keystore + Hardware-backed Keymaster
  - 密钥存储在TEE(Trusted Execution Environment)中
  - SafetyNet/Play Integrity验证设备完整性

#### 安全与性能的权衡

$$\text{Security Score} = f(\text{encryption}, \text{key management}, \text{runtime protection})$$
$$\text{Performance Overhead} = g(\text{decryption latency}, \text{memory overhead})$$

产业实践：通常选择AES-256加密+平台DRM的组合，在安全性和性能间取得平衡。

In [ ]:
class ModelEncryption:
    """AES-256-GCM模型加密/解密"""
    def __init__(self, key: bytes = None):
        if key is None:
            key = os.urandom(32)
        self.key = key

    def encrypt_file(self, src_path: str, dst_path: str) -> dict:
        nonce = os.urandom(12)
        cipher = Cipher(algorithms.AES(self.key), modes.GCM(nonce), backend=default_backend())
        encryptor = cipher.encryptor()
        chunk_size = 1024 * 1024
        t0 = time.perf_counter()
        with open(src_path, 'rb') as fin, open(dst_path, 'wb') as fout:
            fout.write(nonce)
            while True:
                chunk = fin.read(chunk_size)
                if not chunk:
                    break
                fout.write(encryptor.update(chunk))
            encryptor.finalize()
            fout.write(encryptor.tag)
        enc_ms = (time.perf_counter() - t0) * 1000
        src_size = os.path.getsize(src_path)
        dst_size = os.path.getsize(dst_path)
        return {'enc_ms': enc_ms, 'src_size': src_size, 'dst_size': dst_size,
                'overhead_pct': (dst_size - src_size) / src_size * 100}

    def decrypt_file(self, src_path: str, dst_path: str) -> dict:
        t0 = time.perf_counter()
        with open(src_path, 'rb') as fin:
            nonce = fin.read(12)
            fin.seek(-16, 2)
            tag = fin.read(16)
            fin.seek(12)
            cipher = Cipher(algorithms.AES(self.key), modes.GCM(nonce, tag), backend=default_backend())
            decryptor = cipher.decryptor()
            with open(dst_path, 'wb') as fout:
                remaining = os.path.getsize(src_path) - 12 - 16
                while remaining > 0:
                    chunk = fin.read(min(1024*1024, remaining))
                    remaining -= len(chunk)
                    if remaining == 0:
                        fout.write(decryptor.update(chunk))
                    else:
                        fout.write(decryptor.update(chunk))
                decryptor.finalize()
        dec_ms = (time.perf_counter() - t0) * 1000
        return {'dec_ms': dec_ms, 'dst_size': os.path.getsize(dst_path)}

    @staticmethod
    def compute_sha256(filepath: str) -> str:
        h = hashlib.sha256()
        with open(filepath, 'rb') as f:
            while True:
                chunk = f.read(1024*1024)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()

print(f"=== 模型加密实战 (AES-256-GCM) ===")

try:
    from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
    from cryptography.hazmat.backends import default_backend

    tmpdir = tempfile.mkdtemp()
    model_path = os.path.join(tmpdir, 'model.bin')
    enc_path = os.path.join(tmpdir, 'model.bin.enc')
    dec_path = os.path.join(tmpdir, 'model.bin.dec')

    model_data = torch.randn(100, 100).half().numpy().tobytes()
    with open(model_path, 'wb') as f:
        f.write(model_data)

    enc = ModelEncryption()
    enc_info = enc.encrypt_file(model_path, enc_path)
    dec_info = enc.decrypt_file(enc_path, dec_path)

    orig_hash = ModelEncryption.compute_sha256(model_path)
    dec_hash = ModelEncryption.compute_sha256(dec_path)
    integrity = orig_hash == dec_hash

    print(f"\n原始大小: {enc_info['src_size']/1024:.1f} KB")
    print(f"加密大小: {enc_info['dst_size']/1024:.1f} KB (overhead: {enc_info['overhead_pct']:.2f}%)")
    print(f"加密耗时: {enc_info['enc_ms']:.2f} ms")
    print(f"解密耗时: {dec_info['dec_ms']:.2f} ms")
    print(f"完整性校验: {'✓ 通过' if integrity else '✗ 失败'}")
    print(f"SHA256: {orig_hash[:32]}...")

    print(f"\n--- 逐层解密方案（大模型适用） ---")
    layer_weights = {f'layer.{i}.weight': torch.randn(256, 256).half() for i in range(8)}
    layer_enc = ModelEncryption()
    for name, weight in layer_weights.items():
        layer_path = os.path.join(tmpdir, f'{name}')
        with open(layer_path, 'wb') as f:
            f.write(weight.numpy().tobytes())
        enc_result = layer_enc.encrypt_file(layer_path, layer_path + '.enc')
        print(f"  {name}: {weight.shape}, 加密{enc_result['enc_ms']:.2f}ms, overhead={enc_result['overhead_pct']:.1f}%")

    shutil.rmtree(tmpdir, ignore_errors=True)
except ImportError:
    print("cryptography未安装，跳过加密演示 (pip install cryptography)")

print(f"\n=== 产业级加密方案对比 ===")
schemes = [
    ('AES-256-GCM全量加密', '中', '加载+15%', '推理0%', '通用保护'),
    ('逐层AES加密+运行时解密', '中高', '加载+5%', '推理+10%', '内存受限设备'),
    ('iOS Data Protection+Keychain', '高', '加载+2%', '推理0%', 'iOS端侧'),
    ('Android Keystore+TEE', '高', '加载+3%', '推理0%', 'Android端侧'),
]
print(f"\n{'方案':<30} {'安全等级':<8} {'加载开销':<12} {'推理开销':<12} {'适用场景'}")
print("-" * 80)
for name, sec, load, infer, scene in schemes:
    print(f"{name:<30} {sec:<8} {load:<12} {infer:<12} {scene}")

print(f"\n=== 产业实践要点 ===")
print(f"1. AES-GCM提供认证加密: 同时保证机密性和完整性")
print(f"2. 密钥管理是核心: 硬件安全模块(TEE/SE) > 软件混淆 > 硬编码")
print(f"3. 逐层解密适合大模型: 不需要一次性解密全部权重")
print(f"4. 平台DRM是最佳方案: iOS Secure Enclave / Android TEE")
print(f"5. 没有绝对安全: 加密提高攻击成本，配合法律保护")
print(f"6. SHA256校验: 确保解密后模型与原始模型完全一致")

---
### 模型注册中心与版本管理

产业级部署需要管理多个模型版本、支持A/B测试和灰度发布。模型注册中心（Model Registry）是基础设施。

#### 为什么需要模型注册中心？

1. **版本追踪**：每次训练产出带版本号的模型，可追溯任何线上模型对应的训练配置
2. **A/B测试**：新模型先小流量验证效果，逐步放量
3. **回滚能力**：新模型出问题时秒级回滚到上一版本
4. **血缘追踪**：记录模型→数据集→训练配置的完整链路

#### 主流模型注册中心

| 平台 | 特点 | 适用场景 |
|------|------|----------|
| **HuggingFace Hub** | 社区生态，Git-like版本管理 | 开源模型分发 |
| **MLflow Model Registry** | 实验追踪+模型注册+部署 | ML全生命周期管理 |
| **Weights & Biases** | 实验追踪+模型版本+协作 | 团队协作场景 |
| **自建Registry** | S3/MinIO+元数据DB | 企业私有化部署 |

#### 版本管理策略

- **语义版本号**：`v{major}.{minor}.{patch}`，major=架构变更，minor=重训练，patch=微调
- **哈希校验**：每个模型文件附带SHA256哈希，确保完整性
- **元数据嵌入**：训练配置、数据集版本、评估指标等嵌入模型元数据

#### 灰度发布流程

```
新模型v2.0
  │
  ▼ 1%流量
监控指标(PPL/延迟/崩溃率) → 异常? → 回滚v1.9
  │ 正常
  ▼ 10%流量
监控指标 → 异常? → 回滚
  │ 正常
  ▼ 50% → 100%流量
全量发布v2.0
```

In [ ]:
class ModelRegistry:
    """模型注册中心模拟"""
    def __init__(self):
        self.models = {}
        self.deployments = {}

    def register_model(self, name, version, model_hash, size_mb, metrics, config):
        """注册模型版本"""
        key = f"{name}:{version}"
        self.models[key] = {
            'name': name, 'version': version, 'hash': model_hash,
            'size_mb': size_mb, 'metrics': metrics, 'config': config,
            'status': 'registered',
        }

    def deploy_canary(self, name, version, traffic_pct=1):
        """灰度发布"""
        key = f"{name}:{version}"
        if key not in self.models:
            return f"模型{key}未注册"
        self.models[key]['status'] = f'canary({traffic_pct}%)'
        self.deployments[key] = {'traffic_pct': traffic_pct}
        return f"灰度发布{key}，流量{traffic_pct}%"

    def promote(self, name, version, traffic_pct=100):
        """提升流量"""
        key = f"{name}:{version}"
        self.models[key]['status'] = f'production({traffic_pct}%)'
        self.deployments[key] = {'traffic_pct': traffic_pct}
        return f"提升{key}至{traffic_pct}%流量"

    def rollback(self, name, to_version):
        """回滚"""
        for key in list(self.deployments.keys()):
            if key.startswith(f"{name}:") and to_version not in key:
                self.models[key]['status'] = 'rolled_back'
                del self.deployments[key]
        to_key = f"{name}:{to_version}"
        self.models[to_key]['status'] = 'production(100%)'
        self.deployments[to_key] = {'traffic_pct': 100}
        return f"回滚至{to_key}"

registry = ModelRegistry()
registry.register_model('qwen-1.5b', 'v1.0', 'sha256:abc123', 1200,
                        {'ppl': 5.8, 'latency_ms': 25}, {'lr': 2e-5, 'epochs': 3})
registry.register_model('qwen-1.5b', 'v1.1', 'sha256:def456', 1200,
                        {'ppl': 5.5, 'latency_ms': 25}, {'lr': 1e-5, 'epochs': 5})
registry.register_model('qwen-1.5b', 'v2.0', 'sha256:ghi789', 1350,
                        {'ppl': 5.2, 'latency_ms': 28}, {'lr': 3e-5, 'epochs': 3, 'new_data': True})

print("=== 模型注册中心 ===")
for key, info in registry.models.items():
    print(f"\n{key}:")
    print(f"  哈希: {info['hash']}")
    print(f"  大小: {info['size_mb']}MB")
    print(f"  指标: PPL={info['metrics']['ppl']}, 延迟={info['metrics']['latency_ms']}ms")
    print(f"  状态: {info['status']}")

print(f"\n=== 灰度发布流程 ===")
print(registry.deploy_canary('qwen-1.5b', 'v2.0', traffic_pct=1))
print(registry.promote('qwen-1.5b', 'v2.0', traffic_pct=10))
print(registry.promote('qwen-1.5b', 'v2.0', traffic_pct=50))
print(registry.promote('qwen-1.5b', 'v2.0', traffic_pct=100))

print(f"\n=== 回滚演示 ===")
print(registry.rollback('qwen-1.5b', 'v1.1'))
for key, info in registry.models.items():
    print(f"  {key}: {info['status']}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 每个模型版本必须带哈希校验，确保部署的模型未被篡改")
print(f"2. 灰度发布: 1%→10%→50%→100%，每阶段监控关键指标")
print(f"3. 自动回滚: PPL/延迟/崩溃率超过阈值时自动回滚")
print(f"4. 血缘追踪: 记录模型→数据集→训练配置→评估结果的完整链路")
print(f"5. 多端部署: 同一模型版本可能需要为不同平台编译不同格式")

---
### 模型格式转换工具链

产业级部署中，模型需要经历多次格式转换。理解转换工具链和常见陷阱是端侧部署工程师的必备技能。

#### 转换工具链全景

```
                    ┌──→ GGUF (convert-hf-to-gguf.py)
                    ├──→ ONNX (torch.onnx.export)
PyTorch (.pt) ─────┼──→ SafeTensors (safetensors.torch)
                    ├──→ Core ML (coremltools.convert)
                    └──→ TFLite (torch→ONNX→tflite)
                                        │
ONNX ────────┬──→ TensorRT (trtexec)
             ├──→ OpenVINO (mo)
             ├──→ QNN Binary (qnn-onnx-converter)
             └──→ ONNX Runtime (直接加载)
```

#### 常见转换陷阱

| 陷阱 | 症状 | 解决方案 |
|------|------|----------|
| **算子不支持** | 转换报错 | 分解为支持的基础算子 |
| **精度漂移** | PPL异常升高 | 逐层对比余弦相似度 |
| **动态shape丢失** | 推理shape固定 | 检查dynamic_axes设置 |
| **量化格式不兼容** | 跨框架量化结果不同 | 使用目标框架自带量化 |
| **opset版本不匹配** | 新算子缺失 | 升级opset或注册自定义算子 |

#### 精度验证流程

1. **逐层余弦相似度**：$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|} > 0.999$
2. **端到端PPL对比**：转换前后PPL增加应<0.5
3. **下游任务评估**：在标准benchmark上验证任务指标
4. **边界case测试**：极端输入（空字符串、超长序列）的行为一致性

In [ ]:
class ConversionVerifier:
    """模型格式转换精度验证"""
    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.sum(a * b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    @staticmethod
    def verify_tensor_pair(orig: np.ndarray, converted: np.ndarray, name: str) -> dict:
        cos_sim = ConversionVerifier.cosine_similarity(orig.flatten(), converted.flatten())
        max_diff = float(np.max(np.abs(orig - converted)))
        mean_diff = float(np.mean(np.abs(orig - converted)))
        passed = cos_sim > 0.999 and max_diff < 1e-3
        return {'name': name, 'cos_sim': cos_sim, 'max_diff': max_diff,
                'mean_diff': mean_diff, 'passed': passed}

    @staticmethod
    def verify_onnx_conversion(model, onnx_path, test_inputs):
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
            results = []
            for test_input in test_inputs:
                with torch.no_grad():
                    pt_out = model(test_input).numpy()
                ort_out = sess.run(None, {'input': test_input.numpy()})[0]
                result = ConversionVerifier.verify_tensor_pair(pt_out, ort_out, f'shape={list(test_input.shape)}')
                results.append(result)
            return results
        except ImportError:
            return None

print("=== 格式转换精度验证实战 ===")

model = SimpleTransformerBlock(dim=64, n_heads=4)
model.eval()

tmpdir = tempfile.mkdtemp()
onnx_path = os.path.join(tmpdir, 'verify_model.onnx')

dummy = torch.randn(1, 8, 64)
torch.onnx.export(model, dummy, onnx_path, opset_version=17,
                   input_names=['input'], output_names=['output'],
                   dynamic_axes={'input': {0: 'batch', 1: 'seq'}, 'output': {0: 'batch', 1: 'seq'}})

test_inputs = [torch.randn(1, 8, 64), torch.randn(2, 16, 64), torch.randn(1, 32, 64)]
results = ConversionVerifier.verify_onnx_conversion(model, onnx_path, test_inputs)

if results:
    print(f"\nPyTorch → ONNX 转换精度验证:")
    print(f"{'输入shape':<25} {'余弦相似度':<15} {'最大差异':<15} {'平均差异':<15} {'通过'}")
    print("-" * 80)
    for r in results:
        status = '✓' if r['passed'] else '✗'
        print(f"{r['name']:<25} {r['cos_sim']:<15.6f} {r['max_diff']:<15.2e} {r['mean_diff']:<15.2e} {status}")
else:
    print("ONNX Runtime未安装，跳过验证")

print(f"\n--- 量化精度损失模拟 ---")
weight = torch.randn(1024, 1024)
for bits in [8, 6, 5, 4, 3, 2]:
    qmax = 2 ** (bits - 1) - 1
    scale = weight.abs().max() / qmax
    quantized = torch.round(weight / scale) * scale
    cos_sim = ConversionVerifier.cosine_similarity(weight.numpy().flatten(), quantized.numpy().flatten())
    mse = float(torch.mean((weight - quantized) ** 2))
    print(f"  INT{bits}: cos_sim={cos_sim:.6f}, MSE={mse:.6f}")

shutil.rmtree(tmpdir, ignore_errors=True)

print(f"\n=== 转换工具链与风险分析 ===")
conversion_graph = {
    'pytorch': ['onnx', 'safetensors', 'gguf', 'coreml', 'tflite'],
    'onnx': ['tensorrt', 'openvino', 'qnn', 'onnxruntime'],
    'safetensors': ['pytorch', 'gguf'],
}
pitfalls = {
    ('pytorch', 'onnx'): '动态shape、自定义算子、opset版本',
    ('pytorch', 'gguf'): '量化格式选择、词表对齐',
    ('onnx', 'tensorrt'): '算子兼容性、动态shape、精度校准',
    ('onnx', 'qnn'): '算子分解、NPU精度约束',
    ('pytorch', 'coreml'): 'ANE算子兼容性、State API',
}

scenarios = [
    ('pytorch', 'gguf', '端侧CPU推理'),
    ('pytorch', 'tensorrt', 'NVIDIA GPU推理'),
    ('pytorch', 'qnn', '骁龙NPU推理'),
    ('pytorch', 'coreml', 'iOS/macOS推理'),
    ('safetensors', 'tensorrt', 'HuggingFace→GPU'),
]

for source, target, scene in scenarios:
    path = [source]
    current = source
    while current != target:
        next_hop = None
        if target in conversion_graph.get(current, []):
            next_hop = target
        else:
            for intermediate in conversion_graph.get(current, []):
                if target in conversion_graph.get(intermediate, []):
                    next_hop = intermediate
                    break
        if next_hop is None:
            break
        path.append(next_hop)
        current = next_hop
    path_str = ' → '.join(path)
    risk = 'low'
    reasons = []
    for i in range(len(path)-1):
        key = (path[i], path[i+1])
        if key in pitfalls:
            risk = 'medium' if risk == 'low' else 'high'
            reasons.append(f"{path[i]}→{path[i+1]}: {pitfalls[key]}")
    print(f"\n{scene}: {path_str}")
    print(f"  风险: {risk}")
    for r in reasons:
        print(f"  ⚠ {r}")

print(f"\n=== 产业级转换最佳实践 ===")
print(f"1. 最短路径优先: 每次转换都可能引入精度损失")
print(f"2. 逐层验证: 转换后必须逐层对比余弦相似度(cos_sim>0.999)")
print(f"3. 自动化管线: 将转换+验证+打包集成到CI/CD")
print(f"4. 版本锁定: 记录每个转换工具的版本，确保可复现")
print(f"5. 回归测试: 每次框架升级后重新验证转换精度")
print(f"6. 边界case: 测试空输入、超长序列、特殊token等边界情况")

---
### 总结

#### 格式选择决策树

```
目标硬件?
├── NVIDIA GPU → TensorRT (最优性能)
├── Apple Silicon → GGUF (CPU) / Core ML (ANE)
├── Android → TFLite / QNN Binary
├── Intel CPU/NPU → OpenVINO IR
└── 通用CPU → GGUF (llama.cpp)

分发方式?
├── HuggingFace → SafeTensors (安全+mmap)
├── App内嵌 → 加密+平台DRM
└── 私有部署 → 自建Registry+S3
```

#### 产业级最佳实践

1. **SafeTensors是分发标准**：安全、快速、支持分片和按需加载
2. **GGUF是端侧CPU标准**：mmap零拷贝+K-Quant量化，llama.cpp生态
3. **ONNX是转换枢纽**：几乎所有推理引擎都接受ONNX输入
4. **加密是必须**：端侧模型必须加密，平台DRM是最佳方案
5. **版本管理是基础设施**：模型注册中心+灰度发布+自动回滚
6. **转换管线自动化**：CI/CD集成转换+验证+打包，减少人为错误